# Label Coverage Analysis

Determine the optimal sample boundary (Tmag cutoff + plx cutoff) for the Stellar World Model by scanning label coverage rates across brightness and parallax ranges.

Current sample: `processed/df_final.csv`, 6,654 stars (Tmag < 7, plx > 10 mas).  
Current label coverage: rotation=5.4%, flare=2.8%, transit=0.4%, quiet=92.4% — too sparse for meaningful SSL evaluation.

---
## Section 0 — Imports & Setup

In [3]:
import warnings
warnings.filterwarnings('ignore')

import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from astroquery.mast import Catalogs

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

# Project root detection via CLAUDE.md marker
def find_project_root() -> Path:
    p = Path.cwd()
    for _ in range(10):
        if (p / "CLAUDE.md").exists():
            return p
        p = p.parent
    raise FileNotFoundError("CLAUDE.md not found — cannot determine project root")

ROOT = find_project_root()
print(f"Project root: {ROOT}")

# Load reference sample
df_final = pd.read_csv(ROOT / "processed" / "df_final.csv")
print(f"Reference sample (df_final): {len(df_final)} stars  (Tmag < 7, plx > 10 mas)")

Project root: c:\git_repo\Stellar-World-Model
Reference sample (df_final): 6654 stars  (Tmag < 7, plx > 10 mas)


---
## Section 1 — Pull Broad TIC Candidate Set

Query TIC v8 for Tmag < 13, plx > 1 mas. Skip query if `processed/tic_candidates_broad.csv` already exists.

In [4]:
_COARSE_BATCHES = [(0, 7), (7, 8), (8, 9), (9, 10)]
_FINE_BATCHES   = [(round(v * 0.2, 1), round(v * 0.2 + 0.2, 1))
                   for v in range(50, 65)]  # [10.0,10.2) … [12.8,13.0)
TMAG_BATCHES = _COARSE_BATCHES + _FINE_BATCHES

BATCH_COLS = ["ID", "Tmag", "Teff", "logg", "plx", "rad", "mass"]
PROCESSED  = ROOT / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

Catalogs.TIMEOUT = 120

def _batch_csv(lo: float, hi: float) -> Path:
    # Canonical name uses :.1f. Also accept legacy integer-named files (e.g. 0_7)
    # so existing cached batches are recognised without re-querying.
    canonical = PROCESSED / f"tic_candidates_batch_{lo:.1f}_{hi:.1f}.csv"
    legacy    = PROCESSED / f"tic_candidates_batch_{int(lo)}_{int(hi)}.csv"
    if not canonical.exists() and legacy.exists():
        return legacy
    return canonical

def query_batch_with_retry(lo: float, hi: float, max_retries: int = 4, base_wait: int = 10):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            print(f"    attempt {attempt}/{max_retries}...", flush=True)
            # Do NOT pass columns= — astroquery MAST treats unknown kwargs as filter
            # criteria, which causes InvalidQueryError. Filter client-side instead.
            result = Catalogs.query_criteria(
                catalog="TIC",
                Tmag=[lo, hi],
                plx=[1, 9999],
                objType="STAR",
            )
            return result
        except Exception as e:
            last_err = e
            print(f"    {type(e).__name__}: {str(e)[:150]}", flush=True)
            if attempt < max_retries:
                wait = base_wait * attempt
                print(f"    Retrying in {wait}s...", flush=True)
                time.sleep(wait)
    raise last_err

# ── Per-batch query with checkpointing ──────────────────────────────────────
batch_dfs: list[pd.DataFrame] = []
for lo, hi in TMAG_BATCHES:
    csv_path = _batch_csv(lo, hi)
    if csv_path.exists():
        df_b = pd.read_csv(csv_path)
        print(f"  Batch Tmag [{lo:.1f},{hi:.1f}): loaded from {csv_path.name}  ({len(df_b):,} rows)")
        batch_dfs.append(df_b)
        continue

    print(f"  Batch Tmag [{lo:.1f},{hi:.1f}): querying MAST...", flush=True)
    try:
        catalog = query_batch_with_retry(lo, hi)
        df_b = catalog.to_pandas()
        for col in df_b.columns:
            df_b[col] = pd.to_numeric(df_b[col], errors='coerce')
        # Filter to desired columns client-side (columns= kwarg not supported by MAST)
        keep = [c for c in BATCH_COLS if c in df_b.columns]
        df_b = df_b[keep].copy()
        df_b.to_csv(csv_path, index=False)
        print(f"    -> {len(df_b):,} rows  saved to {csv_path.name}")
        batch_dfs.append(df_b)
    except Exception as e:
        print(f"    FAILED after retries: {type(e).__name__}: {e}")
        print(f"    Re-run this cell to resume from this batch.")
        break

# ── Concatenate all completed batches into df_broad ─────────────────────────
df_broad = pd.concat(batch_dfs, ignore_index=True) if batch_dfs else pd.DataFrame(columns=BATCH_COLS)

completed = [f"[{lo:.1f},{hi:.1f})" for lo, hi in TMAG_BATCHES if _batch_csv(lo, hi).exists()]
missing   = [f"[{lo:.1f},{hi:.1f})" for lo, hi in TMAG_BATCHES if not _batch_csv(lo, hi).exists()]
print(f"\ndf_broad: {len(df_broad):,} rows from {len(batch_dfs)} completed batches")
print(f"  Completed : {', '.join(completed) if completed else 'none'}")
if missing:
    print(f"  Missing   : {', '.join(missing)}  <- re-run cell to fetch")

if len(df_broad) > 0:
    print(f"  Tmag range: [{df_broad['Tmag'].min():.2f}, {df_broad['Tmag'].max():.2f}]")
    print(f"  plx range : [{df_broad['plx'].min():.2f}, {df_broad['plx'].max():.2f}]")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(df_broad['Tmag'].dropna(), bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
    axes[0].set_xlabel('Tmag')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Tmag distribution (broad TIC set)')
    axes[1].hist(df_broad['plx'].dropna().clip(upper=200), bins=50, color='darkorange', edgecolor='white', linewidth=0.4)
    axes[1].set_xlabel('Parallax (mas)  [clipped at 200]')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Parallax distribution (broad TIC set)')
    plt.tight_layout()
    plt.show()

  Batch Tmag [0.0,7.0): loaded from tic_candidates_batch_0_7.csv  (39,703 rows)
  Batch Tmag [7.0,8.0): loaded from tic_candidates_batch_7_8.csv  (65,213 rows)
  Batch Tmag [8.0,9.0): loaded from tic_candidates_batch_8_9.csv  (151,900 rows)
  Batch Tmag [9.0,10.0): loaded from tic_candidates_batch_9_10.csv  (337,873 rows)
  Batch Tmag [10.0,10.2): querying MAST...
    attempt 1/4...
    TimeoutError: Timeout limit of 600 exceeded.
    Retrying in 10s...
    attempt 2/4...
    TimeoutError: Timeout limit of 600 exceeded.
    Retrying in 20s...
    attempt 3/4...


KeyboardInterrupt: 